# 📊 Phân tích Xu hướng & Giá cả Hàng hóa Theo Thời gian
## **Dự án: Khai phá Dữ liệu Web (Web Mining)**

### **Mục tiêu:**
Phân tích các xu hướng giá cà phê từ dữ liệu thu thập tự động, nhằm:
1. 📈 Xác định xu hướng giá theo thời gian
2. 🔍 Phát hiện các yếu tố ảnh hưởng đến biến động giá
3. 📊 Trích xuất thông tin quan trọng từ tiêu đề tin tức
4. 📉 Phân tích mô hình mùa vụ và chu kỳ giá
5. 🎯 Đưa ra nhận xét và khuyến nghị

---

**Dữ liệu:** data_cafe_processed.csv (2875 bản ghi, 496 bản ghi có giá)  
**Thời gian:** 12/01/2026 - 18/03/2026 (67 ngày)  
**Giá trung bình:** 100.782 đ/kg | **Phạm vi:** 90.400 - 152.000 đ/kg

In [ ]:
# ==================== IMPORT THƯ VIỆN ====================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Thiết lập ngôn ngữ tiếng Việt cho matplotlib
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
sns.set_style("whitegrid")

# Tăng kích thước hình vẽ
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Thư viện đã được import thành công!")

## 1️⃣ BƯỚC 1: Load & Khám phá Dữ liệu

In [ ]:
# Load dữ liệu đã được xử lý
df = pd.read_csv('data_cafe_processed.csv')

# Chuyển đổi thành datetime
df['time'] = pd.to_datetime(df['time'])
df['date'] = pd.to_datetime(df['date'])

print(f"📊 Tổng số bản ghi: {len(df)}")
print(f"📈 Bản ghi có giá: {df['price'].notna().sum()}")
print(f"✅ Phạm vi thời gian: {df['date'].min()} → {df['date'].max()}")
print(f"\n📋 Cột dữ liệu:")
print(df.columns.tolist())
print(f"\n🔍 Loại dữ liệu:")
print(df.dtypes)
print(f"\n📄 5 hàng dữ liệu đầu tiên:")
df.head()

In [ ]:
# Thống kê cơ bản
print("="*70)
print("📊 THỐNG KÊ CƠ BẢN VỀ GIÁ")
print("="*70)
price_stats = df['price'].describe()
print(price_stats)
print(f"\n📌 Giá cao nhất: {df['price'].max():,.0f} đ/kg ({df.loc[df['price'].idxmax(), 'date']})")
print(f"📌 Giá thấp nhất: {df['price'].min():,.0f} đ/kg ({df.loc[df['price'].idxmin(), 'date']})")
print(f"📌 Độ lệch chuẩn: {df['price'].std():,.0f}")

# Phân bố theo danh mục
print("\n" + "="*70)
print("📋 PHÂN BỐ TIN TỨC THEO DANH MỤC")
print("="*70)
category_dist = df['category'].value_counts()
print(category_dist)

## 2️⃣ BƯỚC 2: Tiền xử lý & Làm sạch Dữ liệu

In [ ]:
# Loại bỏ giá trị NaN
df_price = df[df['price'].notna()].copy()
print(f"✅ Bản ghi có giá: {len(df_price)}")

# Sắp xếp theo thời gian
df_price = df_price.sort_values('date').reset_index(drop=True)

# Tính toán các chỉ số thay đổi
df_price['price_change'] = df_price['price'].diff()
df_price['price_pct_change'] = df_price['price'].pct_change() * 100

# Rolling average (7 ngày)
df_price['price_ma7'] = df_price['price'].rolling(window=7, min_periods=1).mean()
df_price['price_ma14'] = df_price['price'].rolling(window=14, min_periods=1).mean()

print(f"✅ Đã thêm cột: price_change, price_pct_change, price_ma7, price_ma14")
print(f"\n📊 Mẫu dữ liệu sau tiền xử lý:")
print(df_price[['date', 'price', 'price_change', 'price_pct_change', 'price_ma7']].head(10))

## 3️⃣ BƯỚC 3: Phân tích Chuỗi Thời gian (Time Series Analysis)

In [ ]:
# Tổng hợp theongày, tuần, tháng
daily_stats = df_price.groupby('date').agg({
    'price': ['mean', 'min', 'max', 'count'],
    'source': 'nunique'
}).reset_index()
daily_stats.columns = ['date', 'price_mean', 'price_min', 'price_max', 'count', 'sources']

weekly_stats = df_price.set_index('date').resample('W').agg({
    'price': ['mean', 'min', 'max', 'std']
})

monthly_stats = df_price.set_index('date').resample('M').agg({
    'price': ['mean', 'min', 'max', 'std']
})

print("📅 THỐNG KÊ HÀNG NGÀY:")
print(daily_stats.head())
print(f"\n📊 THỐNG KÊ HÀNG TUẦN:")
print(weekly_stats.head())
print(f"\n📆 THỐNG KÊ HÀNG THÁNG:")
print(monthly_stats)

## 4️⃣ BƯỚC 4: Hiển thị Xu hướng Giá (Visualization)

In [ ]:
### 📈 Biểu đồ 1: Giá cà phê theo thời gian (với Moving Average)
fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(df_price['date'], df_price['price'], 'o-', label='Giá hàng ngày', alpha=0.6, linewidth=2, markersize=4)
ax.plot(df_price['date'], df_price['price_ma7'], '-', label='MA-7 (Trung bình 7 ngày)', linewidth=2.5, color='orange')
ax.plot(df_price['date'], df_price['price_ma14'], '-', label='MA-14 (Trung bình 14 ngày)', linewidth=2.5, color='red')

ax.set_xlabel('Ngày tháng', fontsize=12, fontweight='bold')
ax.set_ylabel('Giá (đ/kg)', fontsize=12, fontweight='bold')
ax.set_title('📊 Xu hướng Giá Cà phê (01/2026 - 03/2026)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
### 📊 Biểu đồ 2: Phân bố giá theo hạng (Box Plot)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Box plot theo tháng
df_price['month'] = df_price['date'].dt.to_period('M')
df_price.boxplot(column='price', by='month', ax=ax1)
ax1.set_xlabel('Tháng', fontsize=11, fontweight='bold')
ax1.set_ylabel('Giá (đ/kg)', fontsize=11, fontweight='bold')
ax1.set_title('Phân bố Giá theo Tháng', fontsize=12, fontweight='bold')
plt.sca(ax1)
plt.xticks(rotation=0)

# Distribution plot
ax2.hist(df_price['price'], bins=30, edgecolor='black', alpha=0.7, color='skyblue')
ax2.axvline(df_price['price'].mean(), color='red', linestyle='--', linewidth=2, label=f"Trung bình: {df_price['price'].mean():,.0f}")
ax2.axvline(df_price['price'].median(), color='green', linestyle='--', linewidth=2, label=f"Trung vị: {df_price['price'].median():,.0f}")
ax2.set_xlabel('Giá (đ/kg)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Tần suất', fontsize=11, fontweight='bold')
ax2.set_title('Phân bố Tần suất Giá', fontsize=12, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
### 📉 Biểu đồ 3: Thay đổi Giá hàng ngày (Daily Returns)
fig, ax = plt.subplots(figsize=(16, 6))

colors = ['green' if x > 0 else 'red' for x in df_price['price_change']]
ax.bar(df_price['date'], df_price['price_change'], color=colors, alpha=0.7, width=0.8)
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Ngày tháng', fontsize=12, fontweight='bold')
ax.set_ylabel('Thay đổi giá (đ/kg)', fontsize=12, fontweight='bold')
ax.set_title('📊 Thay đổi Giá Hàng ngày (Xanh: Tăng | Đỏ: Giảm)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Thống kê
up_days = (df_price['price_change'] > 0).sum()
down_days = (df_price['price_change'] < 0).sum()
print(f"📈 Ngày giá tăng: {up_days} ({up_days/(up_days+down_days)*100:.1f}%)")
print(f"📉 Ngày giá giảm: {down_days} ({down_days/(up_days+down_days)*100:.1f}%)")

## 5️⃣ BƯỚC 5: Phân tích Nâng cao - Mô hình Mùa vụ & Tương quan

In [ ]:
### 📅 Phân tích theo Tuần trong tháng
df_price['week_of_month'] = (df_price['date'].dt.day - 1) // 7 + 1
weekly_price = df_price.groupby('week_of_month')['price'].agg(['mean', 'std', 'count'])
print("📅 Giá trung bình theo tuần trong tháng:")
print(weekly_price)

fig, ax = plt.subplots(figsize=(10, 6))
weekly_price['mean'].plot(kind='bar', ax=ax, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Tuần trong tháng', fontsize=11, fontweight='bold')
ax.set_ylabel('Giá trung bình (đ/kg)', fontsize=11, fontweight='bold')
ax.set_title('💹 Giá cà phê theo Tuần trong Tháng', fontsize=12, fontweight='bold')
ax.set_xticklabels(['Tuần 1', 'Tuần 2', 'Tuần 3', 'Tuần 4', 'Tuần 5'], rotation=0)
for i, v in enumerate(weekly_price['mean']):
    ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 6️⃣ BƯỚC 6: Kết luận & Nhận xét (Insights)

In [ ]:
print("="*80)
print("🎯 KẾT LUẬN & NHẬN XÉT CHỦ YẾU")
print("="*80)

insights = f"""
📊 KỲ VỌNG CHÍNH:
1. ⬆️ XU HƯỚNG GIÁ:
   - Giá thấp nhất: {df_price['price'].min():,.0f} đ/kg
   - Giá cao nhất: {df_price['price'].max():,.0f} đ/kg
   - Thay đổi tổng: {df_price['price'].max() - df_price['price'].min():+,.0f} đ/kg ({(df_price['price'].max() - df_price['price'].min()) / df_price['price'].min() * 100:+.1f}%)
   - Biến động (Std): {df_price['price'].std():,.0f} đ/kg ({df_price['price'].std() / df_price['price'].mean() * 100:.1f}% CV)

2. 📈 ĐỘNG LỰC THƯƠNG CHỨNG:
   - Ngày tăng giá: {(df_price['price_change'] > 0).sum()} ngày ({(df_price['price_change'] > 0).sum()/(df_price['price_change'].notna().sum()) * 100:.0f}%)
   - Ngày giảm giá: {(df_price['price_change'] < 0).sum()} ngày ({(df_price['price_change'] < 0).sum()/(df_price['price_change'].notna().sum()) * 100:.0f}%)
   
3. 🔄 THAY ĐỔI TRUNG BÌNH HÀNG NGÀY:
   - Tăng trung bình: +{df_price[df_price['price_change'] > 0]['price_change'].mean():,.0f} đ/kg
   - Giảm trung bình: {df_price[df_price['price_change'] < 0]['price_change'].mean():,.0f} đ/kg

4. 📋 SỐ LIỆU TIN TỨC:
   - Tổng tin tức: {len(df):,} (trong {(df['date'].max() - df['date'].min()).days} ngày)
   - Tin có giá: {len(df_price):,} ({len(df_price)/len(df)*100:.1f}%)
   - Nguồn tin: {df['source'].nunique()} tờ báo khác nhau

5. 🎯 DANH MỤC TIN TÍC NỔIBAट:
   - "Tin về cà phê": {(df['category'] == 'Tin về cà phê').sum()} bản ghi
   - "Giá cà phê (Arabica/Robusta)": {(df['category'] == 'Giá cà phê (Arabica/Robusta)').sum()} bản ghi
   - "Xuất khẩu": {(df['category'] == 'Xuất khẩu').sum()} bản ghi
"""

print(insights)